# DiGress GSG Candidate Generation and Validation

This notebook trains a DiGress discrete graph diffusion model on 154 known genus-9 GSG examples, generates candidate graphs, filters them by structural checks, and validates the structurally valid candidates using ChipFiring.jl.

Pipeline:
1. Load known genus-9 GSGs
2. Convert graphs to PyTorch Geometric format
3. Train DiGress
4. Generate 50 candidate graphs
5. Filter by structural checks: V=16, E=24, trivalent, connected
6. Validate candidates with ChipFiring.jl


In [1]:
!pip install torch torchvision torchaudio
!pip install torch-geometric
!pip install pytorch-lightning==2.0.4
!pip install hydra-core==1.3.2 omegaconf==2.3.0
!pip install wandb==0.15.4 torchmetrics==0.11.4
!pip install networkx matplotlib
!git clone https://github.com/cvignac/DiGress.git
import sys
sys.path.insert(0, '/content/DiGress')
print("Done!")

DEPRECATION: Loading egg at /opt/anaconda3/lib/python3.12/site-packages/DiGress-1.0.1-py3.12.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /opt/anaconda3/lib/python3.12/site-packages/DiGress-1.0.1-py3.12.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /opt/anaconda3/lib/python3.12/site-packages/DiGress-1.0.1-py3.12.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /opt/anaconda3/lib/python3.12/site-packages/DiGress-1.0.1-py3.12.egg is deprecated. pip 24.3 will enforce th

In [4]:
# variables

NUM_NODE_TYPES = 1
NUM_EDGE_TYPES = 3  # {no edge, single, double}
NUM_NODES = 16
TARGET_EDGES = 24

In [5]:
import numpy as np
import networkx as nx
from pathlib import Path

# 1. Define the root path to your 'known_gsgs' directory
# (Update this to your actual path)
root_folder = Path("../known_gsgs").expanduser()

graphs = []

# 2. .rglob("*.txt") searches for all .txt files in the root folder AND all subfolders
for file_path in root_folder.rglob("*.txt"):
    try:
        # Attempt to load the file as a numerical matrix
        adj_matrix = np.loadtxt(file_path, dtype=int)
        
        # Verify it loaded as a 2D matrix (to skip things like readme.txt)
        if adj_matrix.ndim == 2:
            G = nx.from_numpy_array(adj_matrix, create_using=nx.MultiGraph)
            graphs.append(G)
        else:
            print(f"Skipped {file_path.name}: Not a 2D matrix.")
            
    except ValueError:
        # If np.loadtxt fails (e.g., it hits words instead of numbers)
        print(f"Skipped {file_path.name}: Contains non-numeric text.")

print(f"\nSuccessfully loaded {len(graphs)} graphs from all folders.")

# ---
# Convert everything to PyG format
# pyg_graphs = [multigraph_to_pyg(G) for G in graphs]

Skipped readme.txt: Contains non-numeric text.

Successfully loaded 1701 graphs from all folders.


In [6]:
from torch_geometric.data import Data
import torch

def multigraph_to_pyg(G):
    nodes = sorted(G.nodes())
    n = len(nodes)
    X = torch.ones(n, NUM_NODE_TYPES, dtype=torch.float)
    y = torch.zeros([1, 0]).float()
    edge_indices = []
    edge_attrs = []
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            mult = G.number_of_edges(nodes[i], nodes[j])
            if mult > 0:
                edge_indices.append([i, j])
                attr = [0.0] * NUM_EDGE_TYPES
                attr[min(mult, NUM_EDGE_TYPES - 1)] = 1.0
                edge_attrs.append(attr)
    if edge_indices:
        edge_index = torch.tensor(edge_indices, dtype=torch.long).t().contiguous()
        edge_attr = torch.tensor(edge_attrs, dtype=torch.float)
    else:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr = torch.zeros((0, NUM_EDGE_TYPES), dtype=torch.float)
    return Data(x=X, edge_index=edge_index, edge_attr=edge_attr, y=y, n_nodes=n * torch.ones(1, dtype=torch.long))

pyg_graphs = [multigraph_to_pyg(G) for G in graphs]
print(f"Converted {len(pyg_graphs)} graphs to PyG format")

Converted 1701 graphs to PyG format


In [7]:
import os
import random
import torch

random.seed(42)
indices = list(range(len(pyg_graphs)))
random.shuffle(indices)

train_size = int(0.7 * len(indices))
val_size = int(0.15 * len(indices))

train_data = [pyg_graphs[i] for i in indices[:train_size]]
val_data = [pyg_graphs[i] for i in indices[train_size:train_size + val_size]]
test_data = [pyg_graphs[i] for i in indices[train_size + val_size:]]

# Use a relative path by removing the leading slash or adding a dot
base_dir = "./data/gsg"

os.makedirs(f"{base_dir}/raw", exist_ok=True)
os.makedirs(f"{base_dir}/processed", exist_ok=True)

torch.save(train_data, f"{base_dir}/raw/train.pt")
torch.save(val_data, f"{base_dir}/raw/val.pt")
torch.save(test_data, f"{base_dir}/raw/test.pt")

print(f"Split: train={len(train_data)}, val={len(val_data)}, test={len(test_data)}")

Split: train=1190, val=255, test=256


In [8]:
import sys
import os
import torch
from torch_geometric.data import InMemoryDataset

# Add parent directory to path so 'src' can be found
sys.path.append(os.path.abspath('..'))

from src.datasets.abstract_dataset import AbstractDataModule, AbstractDatasetInfos
from src.diffusion.distributions import DistributionNodes

class GSGDataset(InMemoryDataset):
    def __init__(self, split, root="./data/gsg", transform=None, pre_transform=None, pre_filter=None):
        self.split = split
        super().__init__(root, transform, pre_transform, pre_filter)
        
        # Explicitly set weights_only=False here
        self.data, self.slices = torch.load(self.processed_paths[0], weights_only=False)
        
    @property
    def raw_file_names(self):
        return ['train.pt', 'val.pt', 'test.pt']
    
    @property
    def processed_file_names(self):
        return [self.split + '.pt']
    
    def download(self):
        pass
    
    def process(self):
        file_idx = {'train': 0, 'val': 1, 'test': 2}
        
        # Explicitly set weights_only=False here as well
        raw_dataset = torch.load(self.raw_paths[file_idx[self.split]], weights_only=False)
        torch.save(self.collate(raw_dataset), self.processed_paths[0])

class GSGDataModule(AbstractDataModule):
    def __init__(self, cfg):
        self.cfg = cfg
        datasets = {'train': GSGDataset(split='train'), 'val': GSGDataset(split='val'), 'test': GSGDataset(split='test')}
        super().__init__(cfg, datasets)
        self.inner = self.train_dataset
        
    def __getitem__(self, item):
        return self.inner[item]

class GSGDatasetInfos(AbstractDatasetInfos):
    def __init__(self, datamodule, dataset_config):
        self.datamodule = datamodule
        self.name = 'gsg_graphs'
        self.n_nodes = self.datamodule.node_counts()
        self.node_types = torch.tensor([1.0])
        self.edge_types = self.datamodule.edge_counts()
        super().complete_infos(self.n_nodes, self.node_types)

print(f"Train dataset size: {len(GSGDataset(split='train'))}")

Train dataset size: 1190


In [ ]:
import os, sys
import torch

# 1. Point directly to the 'src' folder instead of the project root
# This perfectly mimics what the Colab notebook was doing.
src_path = os.path.abspath('DiGress/src') # If 'src' is inside a DiGress folder, use '../DiGress/src'
if src_path not in sys.path:
    sys.path.insert(0, src_path)
    

from omegaconf import OmegaConf

# 2. Because 'src' is now our root path, we remove the 'src.' prefix from these imports
from diffusion_model_discrete import DiscreteDenoisingDiffusion
from diffusion.extra_features import DummyExtraFeatures, ExtraFeatures
from metrics.abstract_metrics import TrainAbstractMetricsDiscrete 
import pytorch_lightning as pl

cfg = OmegaConf.create({
    'general': {'name': 'gsg_generation', 'wandb': 'disabled', 'gpus': 1, 'resume': None, 'test_only': None,
        'check_val_every_n_epochs': 10, 'sample_every_val': 10, 'val_check_interval': None,
        'samples_to_generate': 10, 'samples_to_save': 10, 'chains_to_save': 0, 'log_every_steps': 50,
        'number_chain_steps': 50, 'final_model_samples_to_generate': 50,
        'final_model_samples_to_save': 10, 'final_model_chains_to_save': 0, 'evaluate_all_checkpoints': False},
    'train': {'batch_size': 16, 'num_workers': 0, 'lr': 2e-4, 'weight_decay': 1e-12, 'n_epochs': 300,
        'save_model': True, 'ema_decay': 0.999, 'optimizer': 'adamw', 'clip_grad': None, 'progress_bar': True, 'seed': 42},
    'model': {'type': 'discrete', 'model': 'graph_tf', 'transition': 'marginal', 'diffusion_steps': 500,
        'diffusion_noise_schedule': 'cosine', 'n_layers': 5, 'extra_features': 'cycles',
        'hidden_mlp_dims': {'X': 128, 'E': 64, 'y': 64},
        'hidden_dims': {'dx': 128, 'de': 64, 'dy': 64, 'n_head': 4, 'dim_ffX': 128, 'dim_ffE': 64, 'dim_ffy': 64},
        'lambda_train': [1.0, 0.0]},
    'dataset': {'name': 'gsg', 'datadir': './data/gsg', 'pin_memory': False, 'remove_h': False}
})

datamodule = GSGDataModule(cfg)
dataset_infos = GSGDatasetInfos(datamodule, cfg.dataset)
extra_features = ExtraFeatures(extra_features_type='cycles', dataset_info=dataset_infos)
domain_features = DummyExtraFeatures()
dataset_infos.compute_input_output_dims(datamodule, extra_features, domain_features)

train_metrics = TrainAbstractMetricsDiscrete()

class DummySamplingMetrics:
    def reset(self): pass
    def forward(self, *a, **k): pass
    def __call__(self, *a, **k): pass

model = DiscreteDenoisingDiffusion(cfg=cfg, dataset_infos=dataset_infos, train_metrics=train_metrics,
    sampling_metrics=DummySamplingMetrics(), visualization_tools=None,
    extra_features=extra_features, domain_features=domain_features)

# trainer = pl.Trainer(max_epochs=cfg.train.n_epochs, accelerator='gpu' if torch.cuda.is_available() else 'cpu',
#     devices=1, log_every_n_steps=10, enable_checkpointing=True)

# Check for both CUDA (NVIDIA) and MPS (Apple Silicon)
if torch.cuda.is_available():
    accel = 'gpu'
elif torch.backends.mps.is_available():
    accel = 'mps'
else:
    accel = 'cpu'

trainer = pl.Trainer(
    max_epochs=cfg.train.n_epochs, 
    accelerator=accel,
    devices=1, 
    log_every_n_steps=10, 
    enable_checkpointing=True
)

print("Training...")
trainer.fit(model, datamodule)
print("Done!")

/opt/anaconda3/lib/python3.12/site-packages/torch/nn/modules/linear.py:124: UserWarning: Initializing zero-element tensors is a no-op
  init.kaiming_uniform_(self.weight, a=math.sqrt(5))
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
/opt/anaconda3/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/logger_connector/logger_connector.py:67: UserWarning: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
  warning_cache.warn(


Marginal distribution of the classes: tensor([1.]) for nodes, tensor([0.8323, 0.1677, 0.0000]) for edges
Training...



   | Name           | Type                            | Params
--------------------------------------------------------------------
0  | train_loss     | TrainLossDiscrete               | 0     
1  | val_nll        | NLL                             | 0     
2  | val_X_kl       | SumExceptBatchKL                | 0     
3  | val_E_kl       | SumExceptBatchKL                | 0     
4  | val_X_logp     | SumExceptBatchMetric            | 0     
5  | val_E_logp     | SumExceptBatchMetric            | 0     
6  | test_nll       | NLL                             | 0     
7  | test_X_kl      | SumExceptBatchKL                | 0     
8  | test_E_kl      | SumExceptBatchKL                | 0     
9  | test_X_logp    | SumExceptBatchMetric            | 0     
10 | test_E_logp    | SumExceptBatchMetric            | 0     
11 | train_metrics  | TrainAbstractMetricsDiscrete    | 0     
12 | model          | GraphTransformer                | 2.5 M 
13 | noise_schedule | PredefinedNoiseScheduleDis

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# 1. Look for the lightning_logs directory
log_dir = "lightning_logs"

if not os.path.exists(log_dir):
    print(f"The '{log_dir}' folder hasn't been created yet. The MPS is likely still warming up!")
else:
    # 2. Find the most recent version folder
    versions = [d for d in os.listdir(log_dir) if d.startswith("version_")]
    
    if not versions:
        print("No training versions found yet.")
    else:
        # Sort numerically to guarantee we get the absolute latest run
        latest_version = sorted(versions, key=lambda v: int(v.split('_')[1]))[-1]
        csv_path = os.path.join(log_dir, latest_version, "metrics.csv")
        
        if not os.path.exists(csv_path):
            print(f"The metrics.csv file hasn't been generated in {latest_version} yet.")
        else:
            try:
                # 3. Read the CSV data
                df = pd.read_csv(csv_path)
                
                # 4. Check if 'train_loss' has been logged yet
                if 'train_loss' in df.columns:
                    # Drop rows where train_loss is empty (NaN)
                    train_data = df.dropna(subset=['train_loss'])
                    
                    if train_data.empty:
                        print(f"CSV created for {latest_version}, but no loss values are recorded yet.")
                    else:
                        # 5. Plot the learning curve
                        plt.figure(figsize=(10, 5))
                        plt.plot(train_data['step'], train_data['train_loss'], color='blue', linewidth=2)
                        
                        plt.title(f"Live Training Loss ({latest_version})", fontsize=14, fontweight='bold')
                        plt.xlabel("Training Steps", fontsize=12)
                        plt.ylabel("Loss", fontsize=12)
                        plt.grid(True, linestyle='--', alpha=0.7)
                        
                        plt.show()
                        
                        # Print the most recent metrics
                        latest_step = int(train_data['step'].iloc[-1])
                        latest_loss = train_data['train_loss'].iloc[-1]
                        print(f"✅ Training is active! Current Step: {latest_step} | Latest Loss: {latest_loss:.4f}")
                else:
                    print("The 'train_loss' column hasn't appeared in the CSV yet. Check back in a minute.")
                    
            except Exception as e:
                print(f"Could not read the CSV file. It might be currently being written to. Error: {e}")

In [ ]:
model.eval()
num_to_generate = 50
print(f"Generating {num_to_generate} candidates...")

with torch.no_grad():
    molecule_list = model.sample_batch(batch_id=0, batch_size=num_to_generate,
        keep_chain=0, number_chain_steps=1, save_final=0, num_nodes=NUM_NODES)

generated_graphs = []
for atom_types, edge_types in molecule_list:
    n = atom_types.shape[0]
    G = nx.MultiGraph()
    G.add_nodes_from(range(1, n + 1))
    for row in range(n):
        for col in range(row + 1, n):
            mult = int(edge_types[row, col].item())
            for _ in range(mult):
                G.add_edge(row + 1, col + 1)
    generated_graphs.append(G)

print(f"Generated {len(generated_graphs)} graphs")

In [ ]:
def graph_to_edge_list(G):
    V, E = G.number_of_nodes(), G.number_of_edges()
    edges = sorted(G.edges())
    return f"Graph(V={V}, E={E}, Edges=[{', '.join(f'({u}, {v})' for u, v in edges)}])"

valid_count = 0
valid_graphs = []
for G in generated_graphs:
    is_valid = (G.number_of_nodes() == 16 and G.number_of_edges() == 24
                and all(G.degree(v) == 3 for v in G.nodes()) and nx.is_connected(G))
    if is_valid:
        valid_count += 1
        valid_graphs.append(G)

print(f"Structurally valid: {valid_count}/{len(generated_graphs)} ({100*valid_count/len(generated_graphs):.1f}%)")

with open("digress_candidates.txt", "w") as f:
    f.write(f"# {len(valid_graphs)} candidates from DiGress trained on {len(graphs)} genus 9 GSGs\n\n")
    for i, G in enumerate(valid_graphs):
        f.write(f'Candidate #{i+1}: "{graph_to_edge_list(G)}"\n')

files.download("digress_candidates.txt")
print(f"Saved and downloaded {len(valid_graphs)} candidates")

In [ ]:
## ChipFiring Validation

The previous section only checks structural validity: `V=16`, `E=24`, trivalent, and connected.

This section checks whether the structurally valid DiGress candidates are actual gonality-savings graphs using the ChipFiring validation code.


In [ ]:
!curl -fsSL https://install.julialang.org | sh -s -- --yes

import os
os.environ["PATH"] = os.path.expanduser("~/.juliaup/bin") + ":" + os.environ["PATH"]

!julia -e 'using Pkg; Pkg.add(["Graphs", "TreeWidthSolver"]); Pkg.add(url="https://github.com/vincentxwang/ChipFiring.jl")'
!pip install juliacall

print("Julia + ChipFiring + juliacall installed.")


In [ ]:
from google.colab import files

print("Upload compute_gonality.jl from the chip-firing-rl repo:")
uploaded = files.upload()


In [ ]:
from juliacall import Main as jl

jl.include("compute_gonality.jl")

to_julia_matrix = jl.seval(
    "py_list -> [Int64(py_list[i][j]) for i in 1:length(py_list), j in 1:length(py_list[1])]"
)

def multigraph_to_matrix(G):
    nodes = sorted(G.nodes())
    n = len(nodes)
    matrix = []
    for i in range(n):
        row = []
        for j in range(n):
            row.append(G.number_of_edges(nodes[i], nodes[j]))
        matrix.append(row)
    return matrix

def validate_gsg(G):
    mat = multigraph_to_matrix(G)
    jl_matrix = to_julia_matrix(mat)
    g = jl.ChipFiringGraph(jl_matrix)

    gon = int(jl.compute_gonality(g))
    genus = int(jl.compute_genus(g))

    if gon <= 4 or genus <= 5 or G.number_of_nodes() <= 5:
        return gon, gon, False

    ugon_approx = gon
    rank = gon

    while rank > 0:
        result = int(
            jl.compute_gonality(
                jl.subdivide(g, 2),
                min_d=rank,
                max_d=rank
            )
        )

        if result == -1:
            ugon_approx = rank + 1
            break

        rank -= 1

    return gon, ugon_approx, gon != ugon_approx

validation_results = []

print(f"Validating {len(valid_graphs)} structurally valid DiGress candidates...")
print("-" * 50)

for i, G in enumerate(valid_graphs):
    print(f"Candidate #{i+1}: computing...", end=" ", flush=True)

    try:
        gon, ugon, is_gsg = validate_gsg(G)
        status = "GSG" if is_gsg else "not GSG"

        validation_results.append({
            "candidate": i + 1,
            "gon": gon,
            "ugon_approx": ugon,
            "is_gsg": is_gsg,
        })

        print(f"gon={gon}, ugon_approx={ugon} → {status}")

    except Exception as e:
        validation_results.append({
            "candidate": i + 1,
            "error": str(e),
        })

        print(f"error: {e}")

print("-" * 50)
print("Validation complete.")
